# 12 — Agent Teams

Dynamic, LLM-routed multi-agent collaboration. Define agents with roles, and a coordinator decides who works, in what order, when to loop back. No manual graph wiring needed.

**What you'll learn:**
1. Create team agents with names and roles
2. Set up a coordinator LLM
3. Run a team on a complex task
4. Inspect round-by-round delegation history
5. Control max rounds for safety

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent, AgentTeam, TeamAgent
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env('bedrock')

## 1. Define Team Members

Each `TeamAgent` has a name, role description, and an actual `Agent` instance that does the work.

In [ ]:
# Define team members with specialized roles
researcher = TeamAgent(
    name="researcher",
    role="Expert at researching topics and gathering key facts. Returns concise bullet points.",
    agent=Agent(llm=llm, prompt="You are a research expert. Provide key facts as concise bullet points."),
    capabilities=["research", "fact-finding", "analysis"],
)

writer = TeamAgent(
    name="writer",
    role="Expert at writing clear, engaging content from research notes. Creates polished prose.",
    agent=Agent(llm=llm, prompt="You are a skilled technical writer. Write clear, engaging content."),
    capabilities=["writing", "editing", "content creation"],
)

reviewer = TeamAgent(
    name="reviewer",
    role="Critical reviewer who checks content for accuracy, clarity, and completeness. Can send work back for revision.",
    agent=Agent(llm=llm, prompt="You are a critical reviewer. Point out issues concisely. Say APPROVED if the content is ready."),
    capabilities=["review", "quality assurance", "feedback"],
)

print(f"Team members: {researcher.name}, {writer.name}, {reviewer.name}")

## 2. Create and Run the Team

The coordinator LLM sees agent descriptions and decides who should work on what. It routes dynamically based on the conversation.

In [ ]:
# Create team with coordinator
team = AgentTeam(
    name="content-team",
    coordinator=llm,                          # LLM that routes work between agents
    agents=[researcher, writer, reviewer],    # available team members
    max_rounds=6,                             # safety limit
)

# Run the team on a task
result = team.run("Write a concise technical overview of WebAssembly (WASM) and why it matters.")

# Show round-by-round delegation history
print("=== Team Delegation Log ===")
for r in result.rounds:
    print(f"\nRound {r.number} [{r.agent}]")
    print(f"  Task: {r.prompt[:100]}...")
    print(f"  Output: {r.output[:150]}...")

print(f"\n=== Final Output ({len(result.rounds)} rounds) ===")
display(Markdown(result.output))